In [1]:
import torch

In [23]:
progress = 0.1
freq_per_band = 42
ramp_steepness = 1

mask = torch.zeros(3 * freq_per_band)
band_start = torch.linspace(0, 1, 3)

for i in range(3):
    lo, hi = i * freq_per_band, (i + 1) * freq_per_band
    alpha = torch.clamp((progress - band_start[i]) * ramp_steepness, 0.0, 1.0)
    mask[lo:hi] = alpha

print(sum(mask), mask.numel())

tensor(4.2000) 126


In [2]:
num_frequencies = 12
sigmas=(1.0, 10.0, 50.0)
d_model = 512

freqs_per_band = num_frequencies // len(sigmas)
freqs_per_band

4

In [3]:
B = torch.cat(
            [torch.randn(2, freqs_per_band) * s for s in sigmas], dim=1
        )
B

tensor([[-3.1105e-01, -1.2273e+00, -8.1019e-01,  1.2025e+00, -3.8601e+00,
         -8.8369e+00,  7.7163e+00, -1.9100e+00, -3.7729e+01,  1.2460e+02,
          7.7349e+01,  8.1101e+01],
        [ 1.7475e+00, -2.0840e+00, -4.0891e-02,  4.8144e-01,  2.3370e+00,
         -1.2714e+01, -1.6647e+01, -3.7337e+00,  4.3021e+01,  1.6075e+01,
          1.9593e+01, -1.0381e+01]])

In [4]:
B.shape

torch.Size([2, 12])

In [5]:
total_freq = B.shape[1]
total_freq

12

In [6]:
grid_size = (32, 32)
gh, gw = grid_size

In [12]:
ys = (torch.arange(gh, dtype=torch.float32) + 0.5) / gh
xs = (torch.arange(gw, dtype=torch.float32) + 0.5) / gw
xs

tensor([0.0156, 0.0469, 0.0781, 0.1094, 0.1406, 0.1719, 0.2031, 0.2344, 0.2656,
        0.2969, 0.3281, 0.3594, 0.3906, 0.4219, 0.4531, 0.4844, 0.5156, 0.5469,
        0.5781, 0.6094, 0.6406, 0.6719, 0.7031, 0.7344, 0.7656, 0.7969, 0.8281,
        0.8594, 0.8906, 0.9219, 0.9531, 0.9844])

In [15]:
grid_y, grid_x = torch.meshgrid(ys, xs, indexing='ij')
grid_x.shape

torch.Size([32, 32])

In [20]:
coords = torch.stack([grid_x.reshape(-1), grid_y.reshape(-1)], dim=-1)  # (N, 2) flatennig
coords.shape

torch.Size([1024, 2])

In [21]:
proj = 2 * torch.pi * coords @ B
proj.shape

torch.Size([1024, 12])

In [23]:
feats = torch.cat([torch.sin(proj), torch.cos(proj)], dim=-1)
feats.shape

torch.Size([1024, 24])

In [24]:
pred = torch.rand(1, 3, 12, 124)
pred.shape

torch.Size([1, 3, 12, 124])

In [32]:
should_be_zero = torch.tensor([[0.2, 0.34], [0.2, 0.34], [0.2, 0.34], [0.2, 0.34], [0.2, 0.34]])
should_be_zero.shape

torch.Size([5, 2])

In [ ]:
pred[:, 0, :, :]

torch.Size([1, 12, 124])

In [39]:
sample = torch.rand(256, 256)
sample.shape

torch.Size([256, 256])

In [38]:
torch.zeros(sample.shape[0], 1, sample.shape[2], sample.shape[3])

tensor([[[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]]]])

In [42]:
import numpy as np
np.where(sample > 0.5)

(array([  0,   0,   0, ..., 255, 255, 255]),
 array([  1,   7,  20, ..., 248, 250, 251]))